# Mental Health in Tech — Modeling

**Goal:** Train and compare classification models to predict whether a tech worker has sought mental health treatment.

**Models:**
- Logistic Regression (baseline)
- Random Forest
- Support Vector Machine (SVM)
- XGBoost

**Metrics:** Accuracy, Precision, Recall, F1, ROC-AUC, Cross-Validation

---
## Table of Contents
1. Import Libraries
2. Load Data
3. Train/Test Split
4. Logistic Regression (Baseline)
5. Random Forest
6. Support Vector Machine
7. XGBoost
8. Model Comparison
9. Feature Importance
10. ROC Curves
11. Conclusions

## 1. Import Libraries

In [ ]:
!pip install xgboost

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Libraries loaded!')

ModuleNotFoundError: No module named 'xgboost'

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/merged_encoded.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['treatment'].value_counts())
print(f'\nClass balance: {df["treatment"].mean()*100:.1f}% positive (sought treatment)')
df.head()

## 3. Train/Test Split

In [ ]:
X = df.drop(columns=['treatment'])
y = df['treatment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (important for LR and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')

In [ ]:
# Helper function for evaluating any model
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, X_full, y_full):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    acc  = accuracy_score(y_te, y_pred)
    auc  = roc_auc_score(y_te, y_prob)
    cv   = cross_val_score(model, X_full, y_full, cv=StratifiedKFold(5), scoring='roc_auc').mean()
    report = classification_report(y_te, y_pred, target_names=['No Treatment', 'Sought Treatment'])
    cm   = confusion_matrix(y_te, y_pred)

    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Accuracy:          {acc:.4f}')
    print(f'  ROC-AUC (test):    {auc:.4f}')
    print(f'  ROC-AUC (5-fold):  {cv:.4f}')
    print(f'\nClassification Report:\n{report}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Confusion Matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['No Treatment', 'Sought Treatment'])
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title(f'{name} — Confusion Matrix', fontweight='bold')

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    axes[1].plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'ROC curve (AUC = {auc:.3f})')
    axes[1].plot([0,1],[0,1], color='gray', linestyle='--', linewidth=1.5, label='Random classifier')
    axes[1].fill_between(fpr, tpr, alpha=0.1, color='#3498db')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{name} — ROC Curve', fontweight='bold')
    axes[1].legend(loc='lower right')

    plt.tight_layout()
    plt.show()

    return {'model': model, 'name': name, 'accuracy': acc, 'roc_auc': auc, 'cv_auc': cv,
            'y_pred': y_pred, 'y_prob': y_prob, 'fpr': fpr, 'tpr': tpr}

results = {}
print('Helper function defined!')

## 4. Logistic Regression (Baseline)

We start with Logistic Regression as a simple, interpretable baseline.  
It assumes a linear decision boundary between classes.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

results['LR'] = evaluate_model(
    'Logistic Regression', lr,
    X_train_scaled, X_test_scaled,
    y_train, y_test,
    X_train_scaled, y_train
)

## 5. Random Forest

Random Forest is an ensemble of decision trees.  
It handles non-linear relationships and provides feature importance scores.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

results['RF'] = evaluate_model(
    'Random Forest', rf,
    X_train, X_test,
    y_train, y_test,
    X, y
)

## 6. Support Vector Machine (SVM)

SVM finds the optimal hyperplane that separates classes.  
It works well for high-dimensional spaces but can be slow on large datasets.

In [ ]:
svm = SVC(probability=True, random_state=42, class_weight='balanced', kernel='rbf')

results['SVM'] = evaluate_model(
    'SVM (RBF Kernel)', svm,
    X_train_scaled, X_test_scaled,
    y_train, y_test,
    X_train_scaled, y_train
)

## 7. XGBoost

XGBoost is a gradient boosting algorithm that builds trees sequentially,  
each correcting the errors of the previous one.

In [ ]:
xgb = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)

results['XGB'] = evaluate_model(
    'XGBoost', xgb,
    X_train, X_test,
    y_train, y_test,
    X, y
)

## 8. Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {
        'Model':      r['name'],
        'Accuracy':   round(r['accuracy'], 4),
        'ROC-AUC':    round(r['roc_auc'], 4),
        'CV ROC-AUC': round(r['cv_auc'], 4),
    }
    for r in results.values()
]).sort_values('ROC-AUC', ascending=False)

print('=== MODEL COMPARISON ===')
print(comparison.to_string(index=False))
print(f'\n🏆 Best model by ROC-AUC: {comparison.iloc[0]["Model"]} ({comparison.iloc[0]["ROC-AUC"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart comparison
x = np.arange(len(comparison))
width = 0.25
axes[0].bar(x - width, comparison['Accuracy'],   width, label='Accuracy',   color='#3498db', alpha=0.85)
axes[0].bar(x,         comparison['ROC-AUC'],    width, label='ROC-AUC',    color='#2ecc71', alpha=0.85)
axes[0].bar(x + width, comparison['CV ROC-AUC'], width, label='CV ROC-AUC', color='#e67e22', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison['Model'], rotation=10)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
axes[0].legend()
for i, row in enumerate(comparison.itertuples()):
    axes[0].text(i - width, row.Accuracy   + 0.005, f'{row.Accuracy:.3f}',   ha='center', fontsize=9)
    axes[0].text(i,         row._3         + 0.005, f'{row._3:.3f}',         ha='center', fontsize=9)
    axes[0].text(i + width, row._4         + 0.005, f'{row._4:.3f}',         ha='center', fontsize=9)

# All ROC curves together
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
for (key, r), color in zip(results.items(), colors):
    axes[1].plot(r['fpr'], r['tpr'], linewidth=2.5, color=color,
                 label=f"{r['name']} (AUC={r['roc_auc']:.3f})")
axes[1].plot([0,1],[0,1], 'k--', linewidth=1.5, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All Models', fontweight='bold', fontsize=13)
axes[1].legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
# Random Forest Feature Importance
rf_model = results['RF']['model']
fi_rf = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

# XGBoost Feature Importance
xgb_model = results['XGB']['model']
fi_xgb = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Random Forest
colors_rf = ['#e74c3c' if v == fi_rf.max() else '#3498db' for v in fi_rf.values]
axes[0].barh(fi_rf.index, fi_rf.values, color=colors_rf, alpha=0.85)
axes[0].set_title('Random Forest — Feature Importance', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Importance Score')
for i, v in enumerate(fi_rf.values):
    if v > 0.02:
        axes[0].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

# XGBoost
colors_xgb = ['#e74c3c' if v == fi_xgb.max() else '#2ecc71' for v in fi_xgb.values]
axes[1].barh(fi_xgb.index, fi_xgb.values, color=colors_xgb, alpha=0.85)
axes[1].set_title('XGBoost — Feature Importance', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Importance Score')
for i, v in enumerate(fi_xgb.values):
    if v > 0.02:
        axes[1].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Feature Importance Comparison (red = most important)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 5 features (Random Forest):')
for feat, val in fi_rf.sort_values(ascending=False).head(5).items():
    print(f'  {feat}: {val:.4f}')

## 10. Cross-Validation Deep Dive

In [ ]:
cv_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
}

for name, model in models_cv.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    cv_results[name] = scores
    print(f'{name}: {scores.round(3)} → mean={scores.mean():.3f} ± {scores.std():.3f}')

# Boxplot
plt.figure(figsize=(10, 5))
plt.boxplot(
    [cv_results[m] for m in models_cv.keys()],
    labels=list(models_cv.keys()),
    patch_artist=True,
    boxprops=dict(facecolor='#3498db', alpha=0.6),
    medianprops=dict(color='red', linewidth=2.5)
)
plt.ylabel('ROC-AUC')
plt.title('5-Fold Cross-Validation ROC-AUC Scores', fontweight='bold', fontsize=13)
plt.ylim(0.6, 1.0)
plt.tight_layout()
plt.show()

## 11. Conclusions

In [3]:
best = comparison.iloc[0]
worst = comparison.iloc[-1]

print('=' * 60)
print('MODELING CONCLUSIONS')
print('=' * 60)

print(f'''
🏆 Best Model: {best['Model']}
   ROC-AUC: {best['ROC-AUC']:.4f}
   CV ROC-AUC: {best['CV ROC-AUC']:.4f}

📊 Why Random Forest performed best:
   - Handles non-linear relationships between features
   - Robust to outliers and noisy data
   - Does not require feature scaling
   - Naturally handles mixed feature types (categorical + numerical)

⚠️  Why SVM performed worst:
   - Sensitive to the scale of features (despite scaling)
   - Default RBF kernel may not fit this data structure
   - Less effective on datasets with many categorical features
   - Computationally expensive, may underfit with default params

🔑 Most Important Features (Random Forest):
   1. work_interfere  — mental health impact on work
   2. Age             — younger workers seek treatment more
   3. family_history  — strong genetic/environmental signal
   4. Country         — cultural differences in stigma
   5. care_options    — awareness of available resources

💡 Key Insight:
   Workers who report that mental health issues interfere
   with their work are far more likely to have sought treatment.
   Company-level factors (benefits, care_options) also play
   a significant role — suggesting that employer support
   encourages treatment-seeking behavior.
''')
print('=' * 60)

NameError: name 'comparison' is not defined